# Chapter 11 — Proxy Pattern
## *Controlling Object Access*

**Intent:** Provide a surrogate or placeholder for another object to **control access** to it.

### Proxy flavors
| Type | Purpose |
|---|---|
| **Remote Proxy** | Represents an object in another address space (RPC, gRPC) |
| **Virtual Proxy** | Creates expensive objects on demand (lazy loading) |
| **Protection Proxy** | Controls access based on permissions |
| **Caching Proxy** | Caches results to avoid repeated expensive calls |

### Book context
A gumball machine monitor service: the real machine is at a remote location.  
A local proxy pretends to be the machine.

## Virtual Proxy — lazy image loading

In [ ]:
from abc import ABC, abstractmethod
import time

class Image(ABC):
    @abstractmethod
    def display(self): ...


class RealImage(Image):
    def __init__(self, filename: str):
        self._filename = filename
        self._load()  # expensive!

    def _load(self):
        time.sleep(0.05)  # simulates disk I/O
        print(f"  [RealImage] Loading '{self._filename}' from disk")

    def display(self):
        print(f"  [RealImage] Displaying '{self._filename}'")


class ImageProxy(Image):
    """Delays RealImage creation until first display()."""
    def __init__(self, filename: str):
        self._filename = filename
        self._real: RealImage | None = None

    def display(self):
        if self._real is None:          # lazy instantiation
            self._real = RealImage(self._filename)
        self._real.display()


print("Creating proxy (no disk access)...")
img = ImageProxy("photo.jpg")
print("First display (loads from disk):")
img.display()
print("Second display (cached):")
img.display()

## Protection Proxy — access control

In [ ]:
class PersonBean(ABC):
    @abstractmethod
    def get_name(self) -> str: ...
    @abstractmethod
    def set_name(self, name: str): ...
    @abstractmethod
    def get_interests(self) -> str: ...
    @abstractmethod
    def set_interests(self, i: str): ...


class Person(PersonBean):
    def __init__(self, name: str):
        self._name      = name
        self._interests = ""

    def get_name(self):         return self._name
    def set_name(self, name):   self._name = name
    def get_interests(self):    return self._interests
    def set_interests(self, i): self._interests = i


class OwnerProxy(PersonBean):
    """Owner can set their own data, but not their hotness rating."""
    def __init__(self, person: Person): self._p = person
    def get_name(self):             return self._p.get_name()
    def set_name(self, name):       self._p.set_name(name)
    def get_interests(self):        return self._p.get_interests()
    def set_interests(self, i):     self._p.set_interests(i)


class NonOwnerProxy(PersonBean):
    """Non-owner can view but cannot change name."""
    def __init__(self, person: Person): self._p = person
    def get_name(self):             return self._p.get_name()
    def set_name(self, name):       raise PermissionError("Cannot set name.")
    def get_interests(self):        return self._p.get_interests()
    def set_interests(self, i):     raise PermissionError("Cannot set interests.")


joe = Person("Joe")
owner_view = OwnerProxy(joe)
owner_view.set_interests("bowling, chess")
print("Owner set interests:", owner_view.get_interests())

guest_view = NonOwnerProxy(joe)
print("Guest sees name:", guest_view.get_name())
try:
    guest_view.set_name("Hacker")
except PermissionError as e:
    print(f"Guest blocked: {e}")

## Caching Proxy — using Python's `functools.lru_cache`

In [ ]:
import functools, time

class WeatherService:
    def get_temperature(self, city: str) -> float:
        time.sleep(0.1)  # simulates API call
        return {"NYC": 72.0, "LA": 85.0}.get(city, 65.0)


class CachingWeatherProxy:
    def __init__(self, service: WeatherService):
        self._service = service
        self._get = functools.lru_cache(maxsize=128)(self._service.get_temperature)

    def get_temperature(self, city: str) -> float:
        return self._get(city)


proxy = CachingWeatherProxy(WeatherService())

start = time.perf_counter()
proxy.get_temperature("NYC")  # slow first call
proxy.get_temperature("NYC")  # fast cached call
elapsed = time.perf_counter() - start
print(f"NYC temp: {proxy.get_temperature('NYC')}°F  — 3 calls in {elapsed:.3f}s")

## Python's `__getattr__` dynamic proxy

In [ ]:
class LoggingProxy:
    """Transparent proxy that logs every method call on the wrapped object."""
    def __init__(self, target):
        object.__setattr__(self, '_target', target)

    def __getattr__(self, name):
        attr = getattr(object.__getattribute__(self, '_target'), name)
        if callable(attr):
            def wrapper(*args, **kwargs):
                print(f"[LOG] {name}({args}, {kwargs})")
                return attr(*args, **kwargs)
            return wrapper
        return attr


proxied_list = LoggingProxy([])
proxied_list.append(1)
proxied_list.append(2)
proxied_list.extend([3, 4])

## Interview cheat-sheet

| Question | Answer |
|---|---|
| Proxy vs Decorator? | Both wrap an object. Proxy **controls access**; Decorator **adds behavior**. Proxy often creates the real subject itself. |
| Proxy vs Adapter? | Adapter changes the interface; Proxy keeps the same interface. |
| Real-world? | ORM lazy loading (SQLAlchemy), HTTP reverse proxies, Java RMI stubs, Python `unittest.mock`. |

**Pattern family:** Structural